# Comparison of Ross Data Output to Halloween Data

## Setup

In [52]:
!pip install tabulate


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
%load_ext dotenv
%dotenv

In [56]:
import sys
import pandas as pd
from pathlib import Path
from sqlalchemy import text
from IPython.display import Markdown

# This allows us to use an absolute import for utils
parent_dir = str(Path().resolve().parents[0])
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from utils.db_helpers import createEngine
from utils.notebook_helpers import scrollable_table

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

### Setup Connections to the Ross Clone database and the Halloween database

In [ ]:
halloween_engine = createEngine("HALLOWEEN")
ross_engine = createEngine("ROSS")

## Comparative Analysis

### General High Level Impressions
In order to get a high level view of deltas and to decide where to drill in, we can compare the row counts for each table in the `npd` schema within Ross's clone database to the row counts for each table in the `npd` schema within the Halloween database. 

#### 1. Define a query to count rows for each table in the `npd` schema

In [ ]:
row_count_query = text("""
WITH    tbl AS (
 SELECT Table_Schema, Table_Name
 FROM   information_schema.Tables
 WHERE  Table_Name NOT LIKE 'pg_%'
        AND Table_Schema IN ('npd')
)
SELECT  Table_Schema AS Schema_Name
,       Table_Name
,       (xpath('/row/c/text()', query_to_xml(format(
          'SELECT count(*) AS c FROM %I.%I', Table_Schema, Table_Name
        ), FALSE, TRUE, '')))[1]::text::int AS Records_Count
FROM    tbl
ORDER   BY Records_Count DESC;
""")

#### 2. Run the query against the Halloween database and Ross's clone database and join the results, to assess potential differences at a high level

In [58]:
halloween_row_count_df = pd.read_sql(row_count_query, con = halloween_engine)
ross_row_count_df = pd.read_sql(row_count_query, con = ross_engine)
row_count_df = halloween_row_count_df.merge(ross_row_count_df, on = ['schema_name', 'table_name'], how = 'outer', suffixes = ['_halloween', '_ross'])
row_count_df.set_index(['schema_name', 'table_name'], inplace=True)

display(scrollable_table(row_count_df))

#### 3. Identify Some High Level Impressions

We would expect the number of records present in the Ross's clone database to be equal to or greater than the number of records in the Halloween database, as the data in Ross's clone database represents an additional month's worth of data as well as data enriched by additional sources. We can run a quick comparison of the record counts to identify whether there are any tables in Ross's clone database that have fewer records than the Halloween database.

In [36]:
display(Markdown(row_count_df.loc[row_count_df['records_count_halloween']>row_count_df['records_count_ross']].to_markdown()))

|                                          |   records_count_halloween |   records_count_ross |
|:-----------------------------------------|--------------------------:|---------------------:|
| ('npd', 'clinical_organization')         |               1.94708e+06 |          1.94704e+06 |
| ('npd', 'individual_to_address')         |               8.4451e+06  |          4.68618e+06 |
| ('npd', 'location')                      |               2.39026e+06 |          0           |
| ('npd', 'location_to_endpoint_instance') |          491119           |          0           |
| ('npd', 'organization_to_address')       |               2.39026e+06 |          1.93288e+06 |
| ('npd', 'provider_to_location')          |               4.16866e+06 |          0           |
| ('npd', 'provider_to_organization')      |               3.25129e+06 |          0           |

## HIGHER PRIORITY:  Tables Missing Data

The following tables have 0 records in Ross's clone:
* Location
* Location_to_endpoint_instance
* Provider_to_organization
* Provider_to_location

Without these tables, the following FHIR endpoints _that are currently populated based on the Halloween data_ cannot be populated based on Ross's clone data:
* /fhir/Location
    * Provides information about where healthcare is practiced
* /fhir/PractitionerRole
    * Provides information about the relationship between Practitioners and Organizations
    * Provides information about the relationship between Practitioners and the Locations where they practice
    * Provides information about the relationship between Practitioners and the FHIR Endpoints that they use in the context of the Organizations that they associate with


#### Location
The location table represents addresses where healthcare is practiced (i.e. physical locations where a patient could go to receive care).

- In NPPES: locations are represented in the practice location file.
- In PECOS: I believe locations are represented in the enrollment reassignment, but may also appear elsewhere

<u>Potential Remedy</u>

Determine which addresses represent physical practice locations (based on the source of the address) and write those addresses to the location table



#### Location_to_endpoint_instance
The location_to_endpoint_instance table represents practice locations and the interoperability endpoints that are utilized at those locations. Since organizations can use different EHR systems for different care settings (e.g. inpatient vs. outpatient care) we do not represent a direct organization to endpoint relationship within the API, but rather a location to endpoint relationship (noting that locations are owned by organizations, and so the relationship is, in effect, an organization to endpoint relationship with a higher level of specificity). Note: we still need to validate that this data model assumption makes sense, as we have continually deferred the "endpoint association question."
- In Lantern: the bundles returned by EHR vendors include addresses associated with endpoints

<u>Potential Remedy</u>

Preserve the endpoint to address associations presented by Lantern, at least as a starting point

#### Provider_to_organization

The provider_to_organization table represents the working relationships between practitioners and the organizations on behalf of which they practice. This is the main value add of NPD above NPPES, which represents practitioners and organizations as independent entities without a working relationship.
* In PECOS: enrollment reassignments indicate that a given provider is practicing on behalf of the organization represented in the reassignment
* In NPPES: If a practitioner shares an exact practice location match with an organization, it is probable that the practitioner practices on behalf of that organization (although we have had back and forth as to whether we can adequately deduce provider_to_organization relationships based on shared practice location). If we can determine our degree of certainty here, this might allow us to represent provider to organization relationships for providers that do not bill Medicare (and therefore would not be included in the PECOS data)

<u>Potential Remedy</u>

Utilize the PECOS reassignment data as a starting point for populating the provider_to_organization table and determine whether some combination of additional attributes might give us enough confidence to utilize the shared practice locations as a proxy for working relationship.

#### Provider_to_location

The provider_to_location table indicates the locations at which a provider practices in the context of their working relationship with a given organization. It is also through a practitioner's association with a location at which they practice that their association with the endpoints that pertain to them flows.

* In PECOS: enrollment reassignments specify location(s)
* In NPPES: (see note above about the potential of using shared practice locations for providers that do not bill Medicare)

<u>Potential Remedy</u>

See note above in the provider_to_organization section.

### Clinical Organization Discrepancies

In [37]:
halloween_clinical_orgs = pd.read_sql('select * from npd.clinical_organization', con = halloween_engine)
ross_clinical_orgs = pd.read_sql('select * from npd.clinical_organization', con = ross_engine)

In [51]:
merged_clinical_orgs = halloween_clinical_orgs.merge(ross_clinical_orgs, on = 'npi', how = 'outer', indicator = True, suffixes = ['_halloween', '_ross'])
halloween_only = list(merged_clinical_orgs[merged_clinical_orgs['_merge']=='left_only']['npi'])
ross_only = list(merged_clinical_orgs[merged_clinical_orgs['_merge']=='right_only']['npi'])
display(Markdown(f"NPIs in Halloween, but not in Ross's clone: {halloween_only}"), Markdown(f"NPIs in Ross's clone, but not in Halloween: {ross_only}"))

NPIs in Halloween, but not in Ross's clone: [1023610292, 1023632619, 1063192433, 1114512423, 1134702061, 1144986803, 1174203863, 1184188104, 1194767731, 1225590177, 1235436593, 1255924346, 1306493820, 1356097794, 1356938146, 1407411945, 1417704685, 1467943571, 1477301182, 1588219711, 1598390270, 1619687795, 1629720313, 1639843675, 1659029841, 1699384719, 1699466482, 1750089546, 1780113191, 1821681941, 1851945158, 1861274722, 1861955262, 1881473098, 1912601402, 1922512847, 1932242401]

NPIs in Ross's clone, but not in Halloween: []

Note: it is not clear from a spot check of those records why they are excluded from Ross's clone.

### Individual to Address Discrepancies and Organization to Address Discrepancies

It is unclear why there are far fewer associations between individuals/organizations and addresses in Ross's clone database and the Halloween databse. I would need to have a better understanding of how organization to address and individual to address associations were determined in each ETL flow.

## LOWER PRIORITY: Hold-over Issues From Halloween Data Load

### Cardinality Expectations

In NPPES and PECOS, multiple names, addresses, and phone numbers are associated with both individuals and organizations, which would lead to the expectation of a 1:many cardinality. Since the multiple addresses and phone numbers would likely be represented in the context of locations, and locations aren't populated, I'll exclude those for now.